# Hallucination Detection Demo

A simple, fully offline heuristic for flagging **unsupported claims** in an assistant's answer given
a retrieved context — the practical, engineering-facing version of "hallucination" from the resume
bullet ("Evaluated responses using LLM and non-LLM metrics including ... hallucination ...").

This notebook builds a lightweight **entity/keyword-overlap checker** using only the Python standard
library and `re` — no ML model, no API key. It's a cheap, deterministic, synchronous-friendly check of
exactly the kind Chapter 01 (`01-llm-evaluation-fundamentals.md`) describes as a "non-LLM metric" —
suitable for running on 100% of live traffic before layering a more expensive LLM-as-judge or
Ragas-style faithfulness check (see `01_ragas_style_metrics_from_scratch.ipynb`) on a sample.

**What this heuristic does:**
1. Extracts "entity-like" tokens from the context (numbers, dollar amounts, percentages, capitalized
   multi-word terms, dates) — the kinds of concrete facts a hallucination typically fabricates or
   alters.
2. Extracts the same kinds of tokens from the generated answer.
3. Flags any entity in the answer that doesn't appear (or doesn't approximately appear) anywhere in
   the context as an **unsupported claim** — a candidate hallucination.

In [1]:
import re
from dataclasses import dataclass, field

print("Imports OK")

Imports OK


## Step 1 — Extracting "entity-like" facts with regex

We look for the categories of facts that most commonly get hallucinated in a financial-services
assistant context: numbers, currency amounts, percentages, and capitalized proper-noun-ish phrases
(policy names, product names, department names).

In [2]:
MONEY_RE = re.compile(r"\$\s?\d[\d,]*(?:\.\d+)?")
PERCENT_RE = re.compile(r"\d+(?:\.\d+)?\s?%")
NUMBER_RE = re.compile(r"(?<![\$%\d.])\b\d[\d,]*(?:\.\d+)?\b(?!\s?%)")
# Two-or-more consecutive capitalized words, e.g. "Gold Savings Plan", "Reserve Bank"
PROPER_PHRASE_RE = re.compile(r"\b(?:[A-Z][a-zA-Z]+(?:\s+[A-Z][a-zA-Z]+)+)\b")


def extract_facts(text: str):
    facts = set()
    facts.update(m.strip() for m in MONEY_RE.findall(text))
    facts.update(m.strip() for m in PERCENT_RE.findall(text))
    facts.update(m.strip() for m in NUMBER_RE.findall(text))
    facts.update(m.strip() for m in PROPER_PHRASE_RE.findall(text))
    return facts


sample_context = (
    "The Gold Savings Plan offers a 3.5% annual interest rate. "
    "Account holders can withdraw up to $2,000 per day. "
    "The plan requires a minimum balance of $500."
)
print(extract_facts(sample_context))

{'The Gold Savings Plan', '$2,000', '3', '000', '3.5%', '$500'}


## Step 2 — Comparing answer facts against context facts

A fact in the answer counts as "supported" if it appears in the context's fact set, either exactly or
as a normalized numeric match (so `$2,000` in the answer matches `$2,000.00` in the context, and
`3.5%` matches `3.5 %`). Anything left over is flagged as **unsupported** — a candidate hallucination.

In [3]:
def normalize_fact(fact: str) -> str:
    # Strip formatting differences: commas, trailing .00, extra spaces, case.
    f = fact.strip().lower()
    f = f.replace(",", "")
    f = re.sub(r"\.00\b", "", f)
    f = re.sub(r"\s+", " ", f)
    return f


def find_unsupported_facts(answer: str, context: str):
    answer_facts = extract_facts(answer)
    context_facts = extract_facts(context)
    context_normalized = {normalize_fact(f) for f in context_facts}

    unsupported = []
    for fact in answer_facts:
        if normalize_fact(fact) not in context_normalized:
            unsupported.append(fact)
    return sorted(unsupported), answer_facts, context_facts


@dataclass
class HallucinationReport:
    answer: str
    supported_facts: set = field(default_factory=set)
    unsupported_facts: list = field(default_factory=list)

    @property
    def is_flagged(self) -> bool:
        return len(self.unsupported_facts) > 0

    @property
    def support_rate(self) -> float:
        total = len(self.supported_facts) + len(self.unsupported_facts)
        if total == 0:
            return 1.0
        return len(self.supported_facts) / total


def check_hallucination(answer: str, context: str) -> HallucinationReport:
    unsupported, answer_facts, _ = find_unsupported_facts(answer, context)
    supported = answer_facts - set(unsupported)
    return HallucinationReport(answer=answer, supported_facts=supported, unsupported_facts=unsupported)

## Worked example 1 — a faithful answer

Every concrete fact in the answer traces back to the context. Expect **no flags**.

In [4]:
faithful_answer = (
    "The Gold Savings Plan offers a 3.5% annual interest rate, and you can withdraw up to "
    "$2,000 per day as long as you maintain the $500 minimum balance."
)

report = check_hallucination(faithful_answer, sample_context)
print("Answer:", report.answer)
print("Supported facts:   ", sorted(report.supported_facts))
print("Unsupported facts: ", report.unsupported_facts)
print(f"Support rate: {report.support_rate:.2f}")
print("FLAGGED as possible hallucination" if report.is_flagged else "NOT flagged (faithful)")

Answer: The Gold Savings Plan offers a 3.5% annual interest rate, and you can withdraw up to $2,000 per day as long as you maintain the $500 minimum balance.
Supported facts:    ['$2,000', '$500', '000', '3', '3.5%', 'The Gold Savings Plan']
Unsupported facts:  []
Support rate: 1.00
NOT flagged (faithful)


## Worked example 2 — a hallucinated answer

The answer introduces a `7%` interest rate and a `Platinum Rewards Program` that never appear in the
context — both should be flagged as unsupported.

In [5]:
hallucinated_answer = (
    "The Gold Savings Plan offers a 7% annual interest rate and automatically enrolls you in the "
    "Platinum Rewards Program. You can withdraw up to $2,000 per day."
)

report = check_hallucination(hallucinated_answer, sample_context)
print("Answer:", report.answer)
print("Supported facts:   ", sorted(report.supported_facts))
print("Unsupported facts: ", report.unsupported_facts)
print(f"Support rate: {report.support_rate:.2f}")
print("FLAGGED as possible hallucination" if report.is_flagged else "NOT flagged (faithful)")

assert report.is_flagged, "This example should be flagged as a hallucination"
print("\nSanity check passed: hallucinated example was correctly flagged.")

Answer: The Gold Savings Plan offers a 7% annual interest rate and automatically enrolls you in the Platinum Rewards Program. You can withdraw up to $2,000 per day.
Supported facts:    ['$2,000', '000', 'The Gold Savings Plan']
Unsupported facts:  ['7%', 'Platinum Rewards Program']
Support rate: 0.60
FLAGGED as possible hallucination

Sanity check passed: hallucinated example was correctly flagged.


## Step 3 — Batch scoring, like a monitoring pipeline would

This is the shape of loop a real-time Tier 1 check (see `04-robustness-adversarial-and-safety-testing.md`
and `99-Interview-QA.md` question 15) would run synchronously on every response before deciding
whether to route it to a heavier LLM-as-judge check.

In [6]:
import pandas as pd

test_cases = [
    {"label": "faithful", "answer": faithful_answer},
    {"label": "hallucinated", "answer": hallucinated_answer},
    {
        "label": "partially_hallucinated",
        "answer": "The Gold Savings Plan offers a 3.5% annual interest rate and a $50 sign-up bonus.",
    },
    {
        "label": "faithful_paraphrase",
        "answer": "With the Gold Savings Plan you earn 3.5% interest per year and may take out as much as $2,000 daily.",
    },
]

rows = []
for case in test_cases:
    r = check_hallucination(case["answer"], sample_context)
    rows.append({
        "label": case["label"],
        "support_rate": round(r.support_rate, 2),
        "flagged": r.is_flagged,
        "unsupported_facts": r.unsupported_facts,
    })

pd.DataFrame(rows)

,label,support_rate,flagged,unsupported_facts
0,faithful,1.00,False,[]
1,hallucinated,0.60,True,"[7%, Platinum Rewards Program]"
2,partially_hallucinated,0.75,True,[$50]
3,faithful_paraphrase,0.80,True,[Gold Savings Plan]


Note the `faithful_paraphrase` case: it should score well because the *numeric* facts (3.5%, $2,000)
still match, even though the wording around them changed completely — this is intentional, and
demonstrates why a purely numeric/entity-based check is more robust to paraphrasing than a naive
string-match approach, while still being far cheaper than an embedding or LLM-based check.

## Limitations (be ready to name these in an interview)

- **Only catches concrete, extractable facts.** This heuristic is blind to hallucinated *reasoning* or
  *qualitative* claims that don't contain numbers or capitalized proper nouns — e.g., "this is
  definitely risk-free" when the context never made that claim wouldn't be flagged here, because
  there's no extractable "fact" token for the regex to catch. A qualitative hallucination like that
  needs an LLM-as-judge or NLI-based check (see `01_ragas_style_metrics_from_scratch.ipynb` for the
  next tier up).
- **False positives from legitimate inference.** If the context says "the account has a $500 minimum"
  and the answer correctly says "you need at least five hundred dollars," the regex-based extractor
  would miss the match entirely because of the word-vs-numeral mismatch — a real system would need
  additional normalization (number-word parsing) or would accept this as an acceptable false-positive
  rate for a cheap Tier 1 filter, escalating flagged cases to a more expensive check rather than
  auto-rejecting them outright.
- **No semantic contradiction detection.** Like the faithfulness notebook, this only checks presence/
  absence, not directional correctness — "the limit is $2,000" vs. "the limit is NOT $2,000" would
  both be marked as supported, since `$2,000` appears in both. Combine with sentiment/negation-aware
  parsing or an LLM check for full coverage.

This tiered design — cheap deterministic check on every response, more expensive semantic check on a
sample or on anything flagged — is the same pattern described in `05-building-a-monitoring-dashboard.md`
and in `99-Interview-QA.md` question 12.